# Week 13 Lab: Data Quality and Cleaning

In this lab you'll practice the techniques from the demo on a small product inventory dataset.
The dataset has the same three problem types you saw in the demo:
- Missing numeric values
- Implausible numeric values
- Orphaned IDs

Most of the code structure is provided. Your job is to fill in the missing pieces.

**Submit this completed notebook to GitHub.**

## Setup

In [1]:
import pandas as pd
import numpy as np

# Supplier reference table
suppliers = pd.DataFrame({
    'supplier_id':   ['S01', 'S02', 'S03', 'S04'],
    'supplier_name': ['NorthBay Goods', 'LakeFront Supply', 'ValleyDist Co.', 'ClearPath Inc.']
})

# Product inventory table
inventory = pd.DataFrame({
    'product_id':   ['P001','P002','P003','P004','P005',
                     'P006','P007','P008','P009','P010',
                     'P011','P012','P013','P014','P015'],
    'product_name': ['Desk Chair','Monitor Stand','Keyboard','USB Hub','Webcam',
                     'Headset','Mouse Pad','Desk Lamp','Cable Organizer','Notebook',
                     'Pen Set','Stapler','File Folder','Whiteboard','Eraser Pack'],
    'supplier_id':  ['S01','S02','S01','S99','S03',
                     'S02','S04','S01','S03','S02',
                     'S12','S04','S03','S01','S02'],
    'price':        [189.99, 45.00, 72.50, np.nan, 58.00,
                     92.00, 12.99, 34.50, 18.75, 8.99,
                     6.50, -14.00, 3.25, 129.00, np.nan],
    'stock_qty':    [24, 8, 15, 42, 0,
                     6, 150, 11, 33, 200,
                     500, 7, 88, 3, 45],
    'category':     ['Furniture','Electronics','Electronics','Electronics','Electronics',
                     'Electronics','Office','Office','Office','Office',
                     'Office','Office','Office','Furniture','Office']
})

print('Suppliers:')
print(suppliers)
print(f'\nInventory: {len(inventory)} rows')
print(inventory)

Suppliers:
  supplier_id     supplier_name
0         S01    NorthBay Goods
1         S02  LakeFront Supply
2         S03    ValleyDist Co.
3         S04    ClearPath Inc.

Inventory: 15 rows
   product_id     product_name supplier_id   price  stock_qty     category
0        P001       Desk Chair         S01  189.99         24    Furniture
1        P002    Monitor Stand         S02   45.00          8  Electronics
2        P003         Keyboard         S01   72.50         15  Electronics
3        P004          USB Hub         S99     NaN         42  Electronics
4        P005           Webcam         S03   58.00          0  Electronics
5        P006          Headset         S02   92.00          6  Electronics
6        P007        Mouse Pad         S04   12.99        150       Office
7        P008        Desk Lamp         S01   34.50         11       Office
8        P009  Cable Organizer         S03   18.75         33       Office
9        P010         Notebook         S02    8.99        2

---
## Step 1: Audit

Profile the dataset before making any changes.

In [2]:
# Count missing values in each column
# This is the same pattern from the demo
print('Missing values per column:')
print(inventory.isnull().sum())

Missing values per column:
product_id      0
product_name    0
supplier_id     0
price           2
stock_qty       0
category        0
dtype: int64


In [3]:
# Get summary statistics for the price column
# Look carefully at the min value
print('Price column statistics:')
print(inventory[['price']].describe())

Price column statistics:
            price
count   13.000000
mean    50.574615
std     58.191740
min    -14.000000
25%      8.990000
50%     34.500000
75%     72.500000
max    189.990000


In [4]:
# Compare supplier IDs in inventory against the valid IDs in suppliers table
inventory_ids = set(inventory['supplier_id'])
valid_ids = set(suppliers['supplier_id'])

print('Supplier IDs in inventory:')
print(sorted(inventory_ids))

print('\nValid supplier IDs:')
print(sorted(valid_ids))

# Compute explicit orphaned ID values so audit findings are unambiguous
orphaned_ids = sorted(inventory_ids - valid_ids)
print('\nOrphaned supplier IDs found:')
print(orphaned_ids)

Supplier IDs in inventory:
['S01', 'S02', 'S03', 'S04', 'S12', 'S99']

Valid supplier IDs:
['S01', 'S02', 'S03', 'S04']

Orphaned supplier IDs found:
['S12', 'S99']


**Write your audit findings here:**
- Missing prices: *There are 2 rows which are missing prices.*
- Implausible prices: *There is 1 clear implausible price, which is P012 (the stapler) and has a price of -$14.*
- Orphaned supplier IDs: *The supplier IDs of S12 and S99 are only present in the Inventory table and NOT the supplier table.*

---
## Step 2: Identify

Find the exact rows with each problem.

In [5]:
# Show rows where price is missing
# Hint: inventory[inventory['price'].____()]
missing_price = inventory[inventory['price'].isnull()]
print('Products with missing price:')
print(missing_price[['product_id', 'product_name', 'price']])

Products with missing price:
   product_id product_name  price
3        P004      USB Hub    NaN
14       P015  Eraser Pack    NaN


In [6]:
# Show rows where price is negative
# A price below $0 is not valid for any product
# Hint: use a boolean condition on the price column
invalid_price = inventory[inventory['price'] < 0]
print('Products with invalid price:')
print(invalid_price[['product_id', 'product_name', 'price']])

Products with invalid price:
   product_id product_name  price
11       P012      Stapler  -14.0


In [8]:
# Find rows with supplier_id values that don't exist in the suppliers table
# Step 1: build a set of valid IDs
valid_suppliers = set(suppliers['supplier_id'])

# Step 2: find rows where supplier_id is NOT in the valid set
# Hint: use ~ and .isin()
orphaned = inventory[~inventory['supplier_id'].isin(valid_suppliers)]
print(f'Products with orphaned supplier_id: {len(orphaned)}')
print(orphaned[['product_id', 'product_name', 'supplier_id']])

Products with orphaned supplier_id: 2
   product_id product_name supplier_id
3        P004      USB Hub         S99
10       P011      Pen Set         S12


---
## Step 3: Decide

Write your fix strategy for each problem type in this cell before writing any code.

**Your strategy:**
*Before I go over my strategy I think it is important to state that before I would implement any of these fixes to the data, I would contact someone in the business who might be able to give the needed context/info to fix these problems. For example, it might be quite simple to ask a sales rep or stock manager what they have listed for prices or suppliers; though this example requires the assumption that they might have more/different information than we do, when it is quite likely that our information comes from the same source as them (and therefore would be the same).*
- Missing price: *In our example, we replace the missing prices with the median value (and NOT the average value) and while I think this can be a good strategy; I do think it can introduce some problems. By replacing the missing values with the median instead of the average we reduce the risk of our filled in values being heavily influenced by outliers - though by including these values we can dilude the dataset away from reality. Personally, depending on what this cleaned dataset is going to be used for, I would recommend staying away from filling in this missing values with the average - as filling in these values could introduce some major problems; for example, consider if this dataset will be used to consider discounts on products, and by introducing values that don't match reality for prices, we apply/give discounts on products that shouldn't be marked down, which results in a loss of profit for the company.*
- Negative price: *I would firstly replace the Negative price with a Null value and THEN perform some basic statistical analysis (STD, Avg, Q1-3, Max, Min, etc), before replacing the negative (now null) value with our newly found Median.*
- Orphaned supplier_id: *Since we do not have any other information about the supplier (besides the supplier_id which matches nothing), it would seem like the best practice would be to replace the supplier_id values with a null or uknown value to better reflect the fact that no information about the supplier is known; as by having a supplier id given, it gives off an impression that there is information about the supplier).*

---
## Step 4: Fix

In [9]:
# Always work on a copy - never modify the original
inventory_clean = inventory.copy()
print('Working on a copy. Shape:', inventory_clean.shape)

Working on a copy. Shape: (15, 6)


In [10]:
# Fix 1: Replace negative price with NaN
# This lets us treat it the same as a missing value
# Hint: inventory_clean.loc[inventory_clean['price'] < 0, 'price'] = ____
# (What did you write in Step 3 for the negative price strategy?)
inventory_clean.loc[inventory_clean['price'] < 0, 'price'] = np.nan

print('Rows with missing price after replacing negative:')
print(inventory_clean[inventory_clean['price'].isnull()][['product_id', 'product_name', 'price']])

Rows with missing price after replacing negative:
   product_id product_name  price
3        P004      USB Hub    NaN
11       P012      Stapler    NaN
14       P015  Eraser Pack    NaN


In [11]:
# Fix 2: Calculate the median price from valid rows only
# Remember: we calculate BEFORE filling so the bad values don't affect the result
price_median = inventory_clean['price'].median()
print(f'Median price (valid rows only): ${price_median:.2f}')

# Now fill all missing prices with the median
inventory_clean['price'] = inventory_clean['price'].fillna(price_median)

print(f'Missing prices after imputation: {inventory_clean["price"].isnull().sum()}')

Median price (valid rows only): $39.75
Missing prices after imputation: 0


In [12]:
# Fix 3: Replace orphaned supplier_id values with 'UNKNOWN'
# Hint: reuse valid_suppliers from Step 2
orphan_mask = ~inventory_clean['supplier_id'].isin(valid_suppliers)
inventory_clean.loc[orphan_mask, 'supplier_id'] = 'UNKNOWN'

print('Supplier ID counts after fix:')
print(inventory_clean['supplier_id'].value_counts())

Supplier ID counts after fix:
supplier_id
S01        4
S02        4
S03        3
UNKNOWN    2
S04        2
Name: count, dtype: int64


---
## Step 5: Verify and Save (Initial)

Re-run the checks from Step 2 on the cleaned copy. The three data-quality checks below should be 0.

Then save your cleaned file. In the Dictionary Lookup section, you'll add supplier names and save a final enriched version.

In [13]:
print('=== VERIFICATION ===')

# Check 1: No missing prices
missing = inventory_clean['price'].isnull().sum()
print(f'Missing prices: {missing}  (expected 0)')

# Check 2: No negative prices
negative = (inventory_clean['price'] < 0).sum()
print(f'Negative prices: {negative}  (expected 0)')

# Check 3: No orphaned supplier IDs (UNKNOWN is intentional, not orphaned)
known_valid = valid_suppliers | {'UNKNOWN'}
still_orphaned = (~inventory_clean['supplier_id'].isin(known_valid)).sum()
print(f'Orphaned supplier IDs remaining: {still_orphaned}  (expected 0)')

=== VERIFICATION ===
Missing prices: 0  (expected 0)
Negative prices: 0  (expected 0)
Orphaned supplier IDs remaining: 0  (expected 0)


In [15]:
# Save the cleaned inventory
# Hint: use .to_csv() with index=False
inventory_clean.to_csv('inventory_clean.csv', index=False)
print('Saved inventory_clean.csv')
print(f'Final shape: {inventory_clean.shape}')

Saved inventory_clean.csv
Final shape: (15, 6)


---
## Dictionary Lookup

Practice building a lookup dictionary - the same pattern you'll use in the assignment.
After this section, save again so your final `inventory_clean.csv` includes the new `supplier_name` column.

In [16]:
# Build a dictionary that maps supplier_id to supplier_name
# Hint: use dict(zip(keys_column, values_column))
supplier_lookup = dict(zip(suppliers['supplier_id'], suppliers['supplier_name']))
print('Supplier lookup dictionary:')
print(supplier_lookup)

Supplier lookup dictionary:
{'S01': 'NorthBay Goods', 'S02': 'LakeFront Supply', 'S03': 'ValleyDist Co.', 'S04': 'ClearPath Inc.'}


In [17]:
# Use .map() to add a supplier_name column to inventory_clean
# Products with UNKNOWN supplier_id will get NaN — that's expected
inventory_clean['supplier_name'] = inventory_clean['supplier_id'].map(supplier_lookup)

print('Inventory with supplier names:')
print(inventory_clean[['product_id', 'product_name', 'supplier_id', 'supplier_name', 'price']].to_string())

Inventory with supplier names:
   product_id     product_name supplier_id     supplier_name   price
0        P001       Desk Chair         S01    NorthBay Goods  189.99
1        P002    Monitor Stand         S02  LakeFront Supply   45.00
2        P003         Keyboard         S01    NorthBay Goods   72.50
3        P004          USB Hub     UNKNOWN               NaN   39.75
4        P005           Webcam         S03    ValleyDist Co.   58.00
5        P006          Headset         S02  LakeFront Supply   92.00
6        P007        Mouse Pad         S04    ClearPath Inc.   12.99
7        P008        Desk Lamp         S01    NorthBay Goods   34.50
8        P009  Cable Organizer         S03    ValleyDist Co.   18.75
9        P010         Notebook         S02  LakeFront Supply    8.99
10       P011          Pen Set     UNKNOWN               NaN    6.50
11       P012          Stapler         S04    ClearPath Inc.   39.75
12       P013      File Folder         S03    ValleyDist Co.    3.25
13 

In [18]:
# Practice safe lookup with .get()
# Try looking up a valid ID and an invalid one
print(supplier_lookup.get('S01', 'Not Found'))   # Should return supplier name
print(supplier_lookup.get('S99', 'Not Found'))   # Should return 'Not Found'

NorthBay Goods
Not Found


---
## Reflection

Answer the following questions in this markdown cell.

1. Why did we replace the negative price with NaN before calculating the median, rather than after?

*The reason we replaced the negative price with NaN before calculating the median, rather than after, is because if we had calculated the median BEFORE replacing the negative value our median would be calculated using data we KNOW is bad, which inturn would "follow down stream" and make our newly found median also bad. By setting the negative price to NaN we ensure that it is NOT used during our calculation for median, better ensuring that our median is built on better than data.*

2. For the orphaned supplier IDs, we used `UNKNOWN` instead of guessing a supplier. What is one situation where guessing might be justified, and one where it definitely would not be?

*A justified situation for where guessing the supplier would be acceptable could be - A situation where all suppliers are known (i.e. you only get supplies from a known/certain list of suppliers) and the product that has the orphaned supplier could only reasonable be from a said supplier; for example, perhaps a restaurant which has soda syrup from an unknown supplier, but the restaurant only gets its soda/beverages from 1 supplier so it would be resonable to guess that the soda syrup comes from the 1 supplier.*

*A situation for where guessing the supplier would NOT be acceptable or justified could be - A situation in which the product could come from a multitude of known and unkown suppliers; for example, perhaps a grocery store which gets products from lots of different suppliers (some are local, some national corporations, some are consistent, some are seasonal, etc) has a large supply of potatoes from an unkown supplier. In our example, perhaps the grocery store gets its potatoes from lots of different sources and these sources consistently change (perhaps due to prices, change in local farmers, availability, or whatever), so there is no real way to guess where this supply of potatoes has come from and as a result would not be wise to guess as there is not enough context to make an educated guess.*

3. What is the difference between `supplier_lookup['S99']` and `supplier_lookup.get('S99', 'Not Found')`? When would you use each?

*The main difference between 'supplier_lookup['S99']' and 'supplier_lookup.get('S99', 'Not Found')' is in the .get() - which by including the function, .get(), you are able to ensure that the lookup won't crash on missing keys. You would use 'supplier_lookup['S99']' when you are certain that the key (or supplier id in our case) that you are searching exists; while you would use 'supplier_lookup.get('S99', 'Not Found')' in situations where you are not sure if the key actually exists/will be found.*